In [ ]:
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt



path = '/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/nlcd_tcc_conus_wgs84_v2025-6_20250101_20251231.tif'

with rasterio.open(path) as src:
    
    # 1. Inspect Metadata
    print(f"Width: {src.width}, Height: {src.height}")
    print(f"Number of bands: {src.count}")
    print(f"Coordinate Reference System (CRS): {src.crs}")
    print(f"Bounding Box: {src.bounds}")
    
    # 2. Read the pixel data into a NumPy array
    # Note: Rasterio bands are 1-indexed (starts at 1, not 0)
    # band1 = src.read(1) 
    
    # Read all bands at once into a 3D NumPy array (Bands, Rows, Columns)
    # all_bands = src.read()

In [ ]:
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

# Downsample before plotting to avoid loading the full-resolution raster.
max_size = 1500
with rasterio.open(path) as src:
    scale = min(max_size / src.width, max_size / src.height)
    width = max(1, int(src.width * scale))
    height = max(1, int(src.height * scale))

    image = src.read(
        1,
        out_shape=(height, width),
        resampling=Resampling.average,
        masked=True,
    )

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(image, cmap='viridis')
ax.set_title('Raster Band 1 (downsampled)')
ax.axis('off')
plt.show()

In [ ]:
rasters = {
    'canopy_cover': '/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/nlcd_tcc_conus_wgs84_v2025-6_20250101_20251231.tif',
    'land_cover': '/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/Annual_NLCD_LndCov_2025_CU_C1V2.tif',
    'FctImp': '/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/Annual_NLCD_FctImp_2025_CU_C1V2.tif'
}

categorical = ['land_cover']

states = { # place ids
    "Ohio": 31,
    "Michigan": 29,
    "Indiana": 20,
    "Pennsylvania": 42,
    "Kentucky": 26,
    "West Virginia": 33
}

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

target_crs = 'EPSG:5070' 

# Create a temporary dictionary to hold new paths so we don't modify 'rasters' during iteration
projected_paths = {}

for raster_key, file_path in rasters.items():
    with rasterio.open(file_path) as src:
        # 1. Calculate the new dimensions and spatial transform
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )
        
        # 2. Copy original metadata and update
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        # Set the correct resampling method based on the data type (e.g., NLCD vs TWI)
        resamp_method = Resampling.nearest if raster_key in categorical else Resampling.bilinear
        
        out_path = f'/kaggle/working/{raster_key}_projected.tiff'

        # 3. Create the new file and reproject
        with rasterio.open(out_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs, # Fixed: must be dst_crs
                    resampling=resamp_method
                )
                
        # Store the new path in the temporary dictionary
        projected_paths[f'{raster_key}_proj'] = out_path

# Safely update the original dictionary after the loop finishes
rasters.update(projected_paths)

In [ ]:
import geopandas as gdp
from rasterio.mask import mask
import osmnx as ox



regions = list(states.keys())
for state in regions: regions[regions.index(state)] = f'{state}, USA'
regions = ox.geocode_to_gdf(regions)
regions

In [ ]:
# for raster in rasters:
#     with rasterio.open(rasters[raster]) as src:
#         if regions.crs != src.crs:
#             print(f'Reprojecting states from {regions.crs} to {src.crs}')
#             regions = regions.to_crs(src.crs)
#             print(f'\n\n{regions.crs}')

In [ ]:
# from pathlib import Path
 
# import geopandas as gpd
# import numpy as np
# import rasterio
# from rasterio.mask import mask
 
# raster_path = rasters['canopy_cover']
# output_dir = Path("/kaggle/working/state_rasters")
# output_dir.mkdir(parents=True, exist_ok=True)
 
# # Census TIGER/Line state polygons; STUSPS contains codes like OH and MI.
# state_boundaries = gpd.read_file(
#     "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"
# )
 
# with rasterio.open(raster_path) as src:
#     selected_states = state_boundaries[
#         state_boundaries["STUSPS"].isin(states.keys())
#     ].to_crs(src.crs)
 
#     for _, state in selected_states.iterrows():
#         clipped, transform = mask(
#             src,
#             [state.geometry],
#             crop=True,
#             filled=False,
#         )
 
#         profile = src.profile.copy()
#         profile.update(
#             height=clipped.shape[1],
#             width=clipped.shape[2],
#             transform=transform,
#             compress="deflate",
#         )
 
#         output_path = output_dir / f"{state['STUSPS']}_tcc.tif"
#         with rasterio.open(output_path, "w", **profile) as dst:
#             # The mask marks pixels outside the state's polygon as invalid.
#             dst.write(clipped.filled(0))
#             valid = ~np.ma.getmaskarray(clipped)
#             dst.write_mask(valid.all(axis=0).astype("uint8") * 255)
 
#         print(f"Saved {output_path}")

In [ ]:
# max_size = 1500
# with rasterio.open('/kaggle/working/state_rasters/PA_tcc.tif') as src:
#     scale = min(max_size / dataset.width, max_size / dataset.height)
#     width = max(1, int(dataset.width * scale))
#     height = max(1, int(dataset.height * scale))

#     image = dataset.read(
#         1,
#         out_shape=(height, width),
#         resampling=Resampling.average,
#         masked=True,
#     )

# fig, ax = plt.subplots(figsize=(10, 10))
# ax.imshow(image, cmap='viridis')
# ax.set_title('Raster Band 1 (downsampled)')
# ax.axis('off')
# plt.show()